In [1]:
import numpy as np
from matplotlib import pyplot as plt
import bilby
from scipy.interpolate import CubicSpline
from scipy.special import erf
import shutil

In [2]:
H0 = 3.240779289444365023237687716352957261e-18 # LAL_H0FAC_SI = 100 km/s/Mpc in SI units
# H0=3.2407792903e-18 # stochastic.m value

# Useful functions

In [3]:
def welch_psd(data,window='hann',nperseg=None,fs=None):
    '''
    Inputs
    * data (assumed to have a length that is a power of 2)
    * window (take to be hann)
    * nperseg = Length of each segment in the PSD (TAvg/deltaT=TAvg*Fs)
    * fs = sampling rate in Hz    
        
    Output
    * f = frequency array [from 0 to f_nyquist by df = 1/TAvg]
    * psd = the psd
    '''
    
    # useful quantities
    NSamples=len(data)
    NAvgs = 2 * int(NSamples / nperseg) - 1 # 50% overlapping
    deltaT = 1/fs
    TAvg = nperseg * deltaT
    stride=int(nperseg/2) # 50% overlapping
    fNyquist = fs/2
    
    # frequency array
    fmin=0
    deltaF=1/TAvg
    fmax = fNyquist
    epsilon=deltaF/100
    freqs = np.arange(fmin,fmax+epsilon,deltaF)    
    
    # psd estimate
    psd=np.zeros(len(freqs))
    
    for nn in range(NAvgs): ## nn goes from 0 to NAvgs-1
        # slice data 
        istart = stride * nn
        iend   = istart + nperseg
        subdata = data[istart:iend]
        
        # window
        subdata = np.hanning(nperseg) * subdata
        
        # fft
        subdataTilde = np.fft.rfft(subdata) * deltaT
        
        # psd = |fft|^2 * 2 / T
        psd = psd + 1/NAvgs * (np.abs(subdataTilde)**2 * 2 / TAvg)
        
    windowFactor =  1/nperseg *  ( np.sum(np.hanning(nperseg)**2) )
    psd=psd/windowFactor

    freqs=freqs[:-1]
    psd=psd[:-1]
    
    return freqs,psd    

In [4]:
def window_factors(N):
    '''
    calculate window factors for a hann window
    '''
    w=np.hanning(N)
    w1w2bar=np.mean(w**2)
    w1w2squaredbar=np.mean(w**4)
    
    w1=w[int(N/2):N]
    w2=w[0:int(N/2)]
    w1w2squaredovlbar=1/(N/2.) * np.sum(w1**2*w2**2)
    
    w1w2ovlbar=1/(N/2.) * np.sum(w1*w2)

    return w1w2bar,w1w2squaredbar,w1w2ovlbar,w1w2squaredovlbar

# data


In [5]:
class TimeSeries:
    def __init__(self,times,data):
        self.times=times
        self.t0=times[0]
        self.deltaT=times[1]-times[0]
        self.Fs=1/self.deltaT
        self.data=data
        
    def window(self):
        return TimeSeries(self.times,np.hanning(len(self.data)) * self.data)
        
    def zero_pad(self,zpf=2):
        NZeros = (zpf-1) * len(self.data)
        new_data=np.append(self.data,np.zeros(NZeros))
        additional_times= np.arange(self.times[-1],
                             self.times[-1]+NZeros*self.deltaT,
                             self.deltaT)
        new_times=np.append(self.times,additional_times)
        return TimeSeries(new_times,new_data)
            
    def window_and_fft(self):
        # window
        ts_w = self.window()
    
        # zero pad
        ts_wz = ts_w.zero_pad(zpf=2)
    
        # fft
        data_tilde = np.fft.rfft(ts_wz.data) * self.deltaT
        
        # construct the frequency array
        fmin=0
        fmax=1/(2*ts_wz.deltaT)
        deltaF=1/(len(ts_wz.data) * ts_wz.deltaT)
        epsilon=deltaF/100
        freqs=np.arange(fmin,fmax+epsilon,deltaF)
        
        return FrequencySeries(freqs,data_tilde)
    
class FrequencySeries:
    def __init__(self,freqs,data):
        self.deltaF=freqs[1]-freqs[0]
        self.freqs=freqs
        self.data=data
            
    def __mul__(self,x):
        if type(x) is float or type(x) is int:
            return FrequencySeries(self.freqs,self.data*x)
        return FrequencySeries(self.freqs,self.data*x.data)
    
    def __truediv__(self,x):
        if type(x) is float or type(x) is int:
            return FrequencySeries(self.freqs,self.data/x)
        return FrequencySeries(self.freqs,self.data/x.data)
        
    def coarse_grain(self,newDeltaF):
        fmin=self.freqs[0]
        fmax=self.freqs[-1]
        epsilon=deltaF/100.
        new_freqs=np.arange(fmin,fmax,newDeltaF)
        new_data_real=np.zeros(len(new_freqs))
        new_data_imag=np.zeros(len(new_freqs))
        NBinsToCombine = int(newDeltaF/self.deltaF)
                
        # hack to make coarse graining work
        # add zeros to edges of original frequency series
        zero_padded_freq_min = self.freqs[0] - newDeltaF/2
        zero_padded_freq_max = self.freqs[-1] + newDeltaF/2
        zero_padded_freqs = np.arange(zero_padded_freq_min,
                                     zero_padded_freq_max+epsilon,
                                     self.deltaF)

        zero_padded_data = np.insert(self.data,0,np.zeros(int(NBinsToCombine / 2)))
        zero_padded_data = np.append(zero_padded_data,np.zeros(int(NBinsToCombine / 2)))

        zero_padded_frequency_series = FrequencySeries(zero_padded_freqs,
                                                      zero_padded_data)
      
        for ii in range(len(new_freqs)):          
            istart = ii * NBinsToCombine 
            iend = istart + NBinsToCombine
            
            new_data_real[ii] = np.mean(np.real(
                zero_padded_frequency_series.data[istart:iend]))
            new_data_imag[ii] = np.mean(np.imag(
                zero_padded_frequency_series.data[istart:iend]))

        return FrequencySeries(new_freqs,new_data_real+1j*new_data_imag)
    
def slice_time_series(timeseries,istart,iend):
    return TimeSeries(timeseries.times[istart:iend],
                      timeseries.data[istart:iend])
    

In [6]:
def calc_rho1(N):
    w1w2bar,_,w1w2ovlbar,_=window_factors(100000)
    rho1 = (0.5 * w1w2ovlbar / w1w2bar)**2
    return rho1
    
def calc_bias(segmentDuration,deltaF,deltaT):
    N=int(segmentDuration/deltaT)
    rho1=calc_rho1(N)
    Nsegs=(2 * segmentDuration * deltaF - 1)
    wfactor = (1+2*rho1)**(-1)
    Neff = 2*wfactor*(2*segmentDuration*deltaF-1)
    bias=Neff/(Neff-1)
    return bias

# stochastic pipeline

In [7]:
def stochastic(d1,d2,segmentDuration,deltaF,verbose=True,doOverlap=True,alpha=0,fref=1,orf_file=None):
    
    # error checking
    assert d1.deltaT == d2.deltaT
    
    # compute length of segment and interval
    length_of_segment = int(segmentDuration / d1.deltaT)
    length_of_interval = int(3 * length_of_segment)
            
    if doOverlap:
        # overlapping intervals
        # see https://git.ligo.org/stochastic-public/stochastic/-/blob/master/CrossCorr/src_cc/loadAuxiliaryInput.m/#L288
        bufferSecs = 0 
        jobDuration = d1.times[-1] - d1.times[0] + d1.deltaT
        # number of non-overlapping segments
        M = int( np.floor( (jobDuration - 2*bufferSecs)/segmentDuration ) )
        numSegmentsTotal = 2*M-1
        numIntervalsTotal = 2*(M-2)-1
        intervalTimeStride = int(segmentDuration/2)
        
        stride = int(intervalTimeStride / d1.deltaT)
    else:
        numIntervalsTotal = int(len(d1.data) / length_of_interval)
        stride = length_of_interval

    # initialize point estimate and error bar arrays
    Ys=np.zeros(numIntervalsTotal)
    sigs=np.zeros(numIntervalsTotal)
    Ys_a=np.zeros(numIntervalsTotal)
    sigs_a=np.zeros(numIntervalsTotal)
    
    # initialize Y(f,t) and variance(f,t) 2d-arrays
    # also define frequency and start time arrays
    d1_test=slice_time_series(d1,0,length_of_segment)
    segmentStartTimes=d1.times[2*stride:-4*stride+1:stride]
    freqs,P1_test = welch_psd(d1_test.data,
                            window='hann',
                            nperseg=int(d1_test.Fs/deltaF),
                            fs=d1_test.Fs)
    
    numFreqs = len(freqs)
    
    # intialize arrays
    Y_fs = np.zeros((numFreqs,numIntervalsTotal))
    var_fs = np.zeros((numFreqs,numIntervalsTotal))
    Y_fs_a = np.zeros((numFreqs,numIntervalsTotal))
    var_fs_a = np.zeros((numFreqs,numIntervalsTotal))
    
    # initialize orf, set to 1 if no file provided
    if orf_file is not None:
        f_orf,orf=np.loadtxt(orf_file,unpack=True)
        orfSpline=CubicSpline(f_orf,orf)
        orf = orfSpline(freqs)
    else:
        orf=np.ones(freqs.shape)
    
    # window factors
    w1w2bar,w1w2squaredbar,_,_=window_factors(int(segmentDuration/d1.deltaT))
    
    # loop over intervals
    for II in range(numIntervalsTotal):
        
        offset = II * stride
        
        # split interval into segments 1, 2, 3
        d1_left=slice_time_series(d1,offset,offset+length_of_segment)
        d1_mid =slice_time_series(d1,offset+length_of_segment,offset+2*length_of_segment)
        d1_right=slice_time_series(d1,offset+2*length_of_segment,offset+3*length_of_segment)
        
        d2_left=slice_time_series(d2,offset,offset+length_of_segment)
        d2_mid =slice_time_series(d2,offset+length_of_segment,offset+2*length_of_segment)
        d2_right=slice_time_series(d2,offset+2*length_of_segment,offset+3*length_of_segment)
    
        # window, zero pad, fft
        d1_mid_tilde = d1_mid.window_and_fft()
        d2_mid_tilde = d2_mid.window_and_fft()
        
        # compute cross correlation d1*d2 [segment 2]
        c = FrequencySeries(d1_mid_tilde.freqs, 
                            d1_mid_tilde.data * np.conj(d2_mid_tilde.data))
    
        # coarse grain
        c_cg=c.coarse_grain(deltaF)
    
        # compute power spectra [segment 1 + 3]
        freqs,P1_left = welch_psd(d1_left.data,
                            window='hann',
                            nperseg=int(d1_left.Fs/deltaF),
                            fs=d1_left.Fs)
        _,P1_right = welch_psd(d1_right.data,
                            window='hann',
                            nperseg=int(d1_right.Fs/deltaF),
                            fs=d1_right.Fs)  
        P1 = FrequencySeries(freqs,
                            0.5*(P1_left+P1_right))

        _,P2_left = welch_psd(d2_left.data,
                            window='hann',
                            nperseg=int(d2_left.Fs/deltaF),
                            fs=d2_left.Fs)
        _,P2_right = welch_psd(d2_right.data,
                            window='hann',
                            nperseg=int(d2_right.Fs/deltaF),
                            fs=d2_right.Fs)  
        P2 = FrequencySeries(freqs,
                            0.5*(P2_left+P2_right))        
        
        # construct cross-correlation spectra        

        # proper filtering for Y_f (cross-correlation estimator at each frequency; used for bayesian inference)
        S0 = FrequencySeries(freqs, (3*H0**2)/(10*np.pi**2*freqs**3))
        S_alpha = FrequencySeries(freqs, S0.data * (freqs/fref)**alpha )

        # alpha=0
        Y_f = FrequencySeries(freqs, 2*c_cg.data/(segmentDuration * orf * S0.data ))
        Y_f = Y_f / w1w2bar
        var_f = FrequencySeries(freqs, (P1.data*P2.data) / (2*segmentDuration*deltaF)* orf**(-2) * S0.data**(-2))
        var_f = var_f * w1w2squaredbar / w1w2bar**2

        # alpha potentially non-zero
        Y_f_a = FrequencySeries(freqs, 2*c_cg.data/(segmentDuration * orf * S_alpha.data ))
        Y_f_a = Y_f_a / w1w2bar
        var_f_a = FrequencySeries(freqs, (P1.data*P2.data) / (2*segmentDuration*deltaF)* orf**(-2) * S_alpha.data**(-2))
        var_f_a = var_f_a * w1w2squaredbar / w1w2bar**2

        # for broadband point estimate which assume a powerlaw spectrum with spectral index alpha
        Ys[II],sigs[II] = calc_Y_sigma_from_Yf_varf(Y_f.data[1:],var_f.data[1:], freqs=freqs[1:], alpha=alpha, fref=fref) 
        # alternatively (should give same answer)
        Ys_a[II],sigs_a[II] = calc_Y_sigma_from_Yf_varf(Y_f_a.data[1:],var_f_a.data[1:], freqs=None) 
        
        # narrow band estimator
        Y_fs[:,II],var_fs[:,II] = Y_f.data, var_f.data        
        Y_fs[0,II]=0 # handle zero frequency
        var_fs[0,II]=np.inf

        Y_fs_a[:,II],var_fs_a[:,II] = Y_f_a.data, var_f_a.data        
        Y_fs_a[0,II]=0 # handle zero frequency
        var_fs_a[0,II]=np.inf
        
        if verbose:
            print('stochastic: Done with Interval %u / %u'%(II+1,numIntervalsTotal))
            print('\tY       = %e'%Ys[II])
            print('\tY_a     = %e'%Ys_a[II])
            print('\tsigma   = %e'%sigs[II])
            print('\tsigma_a = %e'%sigs_a[II])
            print('\tSNR     = %f'%(Ys[II]/sigs[II])) 
            print('\tSNR_a     = %f'%(Ys_a[II]/sigs_a[II])) 
        
    return Ys,sigs,Y_fs,var_fs,Ys_a,sigs_a,Y_fs_a,var_fs_a,segmentStartTimes,freqs

In [8]:
def postprocessing(Ys,sigs,jobDuration,segmentDuration,deltaF,deltaT,bufferSecs=0):

    '''
    optimally combine broadband estimators over segments following lazzarini-romano
    '''
    
    M = int( np.floor( (jobDuration - 2*bufferSecs)/segmentDuration ) )
    numSegmentsTotal = 2*M-1
    
    N=int(segmentDuration/deltaT)
    _,w1w2squaredbar,_,w1w2squaredovlbar=window_factors(N)

    k = w1w2squaredovlbar/w1w2squaredbar
    
    Y_odds = Ys[::2]
    Y_evens = Ys[1::2]
    sig_odds = sigs[::2]
    sig_evens = sigs[1::2]
    
    Y_o = np.sum(Y_odds / sig_odds**2) / np.sum(1/sig_odds**2)
    s_o = np.sqrt(1/np.sum(1/sig_odds**2))
    
    Y_e = np.sum(Y_evens / sig_evens**2) / np.sum(1/sig_evens**2)
    s_e = np.sqrt(1/np.sum(1/sig_evens**2))
    
    sig_m2 = 1./s_o**2 + 1./s_e**2 -0.5*(1/sigs[0]**2 + 1/sigs[-1]**2)
    s_oe = np.sqrt( k * 0.5 * s_o**2 * s_e**2 * sig_m2)
                    
    C_oo = s_o**2
    C_oe = s_oe**2
    C_eo = C_oe
    C_ee = s_e**2
    detC = C_oo*C_ee - C_oe**2
    
    lambda_o = (s_e**2 - s_oe**2) / detC
    lambda_e = (s_o**2 - s_oe**2) / detC
    
    lambda_sum = lambda_o + lambda_e
    
    Y_opt = (lambda_o * Y_o + lambda_e * Y_e) / lambda_sum
    var_opt = 1 / lambda_sum
    sig_opt=np.sqrt(var_opt)
    
    bias=calc_bias(segmentDuration,deltaF,deltaT)
    sig_opt=sig_opt*bias

    return Y_opt,sig_opt

In [9]:
def postprocessing_spectra(Y_fs,var_fs,Ys,sigs,jobDuration,segmentDuration,deltaF,deltaT,bufferSecs=0):

    '''
    combine narrow-band estimators over segments, following pygwb
    
    '''
    
    M = int( np.floor( (jobDuration - 2*bufferSecs)/segmentDuration ) )
    numSegmentsTotal = 2*M-1
    
    N=int(segmentDuration/deltaT)
    _,w1w2squaredbar,_,w1w2squaredovlbar=window_factors(N)

    k = w1w2squaredovlbar/w1w2squaredbar

    # broadband quantities
    '''
    sig_odds = sigs[::2]
    sig_evens = sigs[1::2]
    
    s_o = np.sqrt(1/np.sum(1/sig_odds**2))
    s_e = np.sqrt(1/np.sum(1/sig_evens**2))
    sig_m2 = 1./s_o**2 + 1./s_e**2 -0.5*(1/sigs[0]**2 + 1/sigs[-1]**2)
    s_oe = np.sqrt( k * 0.5 * s_o**2 * s_e**2 * sig_m2)
    '''
        
    # narrowband quantities
    Y_fs_odds = Y_fs[:,::2]
    Y_fs_evens = Y_fs[:,1::2]
              
    var_fs_odds = var_fs[:,::2]
    var_fs_evens = var_fs[:,1::2]
    
    X_f_o = np.sum(Y_fs_odds/var_fs_odds,axis=1) 
    X_f_e = np.sum(Y_fs_evens/var_fs_evens,axis=1) 
    
    v_f_o = 1./np.sum(1./var_fs_odds,axis=1)
    v_f_e = 1./np.sum(1./var_fs_evens,axis=1)    
    sig_m2_f = 1./v_f_o + 1./v_f_e - 0.5*(1./var_fs[:,0] + 1./var_fs[:,-1])

    # narrow band estimators for odd and even segments
    Y_f_o = X_f_o * v_f_o
    Y_f_e = X_f_e * v_f_e
    
    # sum over frequencies (ignoring f=0)
    v_o = 1./np.sum(1./v_f_o[1:])
    v_e = 1./np.sum(1./v_f_e[1:])
    v_1 = 1./np.sum(1/var_fs[1:,0])
    v_N = 1./np.sum(1/var_fs[1:,-1])
    sig_m2 = 1./v_o + 1./v_e - 0.5*(1./v_1 + 1./v_N)

    Y_f = ( X_f_o*(1- 0.5*k*v_o * sig_m2) + X_f_e*(1. - 0.5*k*v_e * sig_m2)) / (1./v_f_o + 1./v_f_e - k*sig_m2_f)     
    var_f = (1. - 0.25*k**2 * v_o * v_e * sig_m2**2) / (1./v_f_o + 1./v_f_e - k*sig_m2_f)

    '''
    Y_f = ( X_f_o*(1- 0.5*k*s_o**2 * sig_m2) + X_f_e*(1. - 0.5*k*s_e**2 * sig_m2))/(1./v_f_o + 1./v_f_e - k*sig_m2_f)     
    var_f = (1. - 0.25*k**2 * s_o**2 * s_e**2 * sig_m2**2) / (1./v_f_o + 1./v_f_e - k*sig_m2_f)
    '''
    
    bias=calc_bias(segmentDuration,deltaF,deltaT)
    var_f=var_f*bias**2

    '''
    print('broadband v_o=', s_o**2)
    print('narowband v_o=', v_o)
    print('broadband v_e=', s_e**2)
    print('narowband v_e=', v_e)
    print('broadband sigma_m2=', sig_m2)
    print('narowband sigma_m2=', sig_m2_nb)
    '''

    return Y_f, var_f

In [10]:
def calc_Y_sigma_from_Yf_varf(Y_f,var_f,freqs=None,alpha=0,fref=1):
    '''
    combine narrowband estimators (already combined over segments) to calculate broadband estimators (reweighting by alpha if desired)
    '''
    
    if freqs is not None:
        weights = (freqs/fref)**alpha
    else:
        weights=np.ones(Y_f.shape)
    
    var = 1 / np.sum(var_f**(-1) * weights**2)
    
    #Y = np.sum(Y_f * var_f**(-1)) / np.sum( var_f**(-1) )
    Y = np.sum(Y_f * weights * (var/var_f) ) 
    
    sigma=np.sqrt(var)

    ##############################################
    # alternate calculation
    '''
    Y_ref_f = Y_f * weights
    var_ref_f = var_f * weights**2

    Y = np.sum(Y_ref_f/var_ref_f)/np.sum(1./var_ref_f)
    var = 1/np.sum(1./var_ref_f)
    sigma = np.sqrt(var)
    '''
    
    return Y,sigma

In [11]:
def cleanup_dir(outdir):
    # cleanup
    try:
        shutil.rmtree(outdir)
    except OSError as e:
        pass # directory doesn't exist

In [12]:
class Baseline():
    def __init__(self,name,Y_f,var_f,freqs,calibration_epsilon=0):
        """
        Y_f: array_like
            Y(f) (combined over time, remove zero frequency first)
        var_f: array_like
            var(f) (combined over time, remove zero frequency first)
        freqs: array_like
            frequency array (remove zero frequency first)
        calibration_epsilon: number
            calibration uncertainty for this baseline
        """
        self.name=name
        self.Y_f = np.real(Y_f) # only want real part for PE
        self.var_f = var_f
        self.freqs = freqs
        self.calibration_epsilon = calibration_epsilon
        
        # sanitize input
        finite_variance_filter=np.logical_not(np.isinf(var_f))
        self.freqs=self.freqs[finite_variance_filter]
        self.Y_f=self.Y_f[finite_variance_filter]
        self.var_f=self.var_f[finite_variance_filter]

class GWBLikelihood(bilby.Likelihood):
    def __init__(self, baselines,parameters=None):
        """
        A parent class for GWB data analysis with Bilby

        Parameters
        ----------
        baselines: a list of Baseline objects containing cross correlation data.
        """
        super().__init__(parameters=parameters)
        self.baselines=baselines

    def log_likelihood_IJ(self,baseline,noise=False):
        if noise:
            Y_model_f=0
        else:
            Y_model_f = self.OmegaGW(baseline.freqs)
            
        # simple likelihood without calibration uncertainty    
        if baseline.calibration_epsilon==0: 
            logL_IJ= -0.5 * ( np.sum((baseline.Y_f-Y_model_f)**2 / baseline.var_f) +
                        np.sum(np.log(2 * np.pi * baseline.var_f)) )
            
        # likelihood with calibration uncertainty marginalizatione done analytically
        # see https://stochastic-alog.ligo.org/aLOG//index.php?callRep=339711
        # note \cal{N} = \Prod_j sqrt(2*pi*sigma_j^2)
        else: 
            A = (baseline.calibration_epsilon)**(-2) + np.sum(Y_model_f**2/baseline.var_f) 
            B = (baseline.calibration_epsilon)**(-2) + np.sum(Y_model_f*baseline.Y_f/baseline.var_f)
            C = (baseline.calibration_epsilon)**(-2) + np.sum(baseline.Y_f**2/baseline.var_f)
            log_norm = -0.5*np.sum(np.log(2*np.pi*baseline.var_f))
            
            logL_IJ = (log_norm 
                        - 0.5*np.log(A*baseline.calibration_epsilon**2)
                        + np.log(1+erf(B/np.sqrt(2*A))) 
                        - np.log(1+erf(1/np.sqrt(2*baseline.calibration_epsilon**2)))
                        - 0.5*(C-B**2/A)
                      )
            
        return logL_IJ
  
    def log_likelihood(self):
        logL=0
        for baseline in self.baselines: 
            logL= logL + self.log_likelihood_IJ(baseline,noise=False)
        return logL
    
    def noise_log_likelihood(self):
        logL=0
        for baseline in self.baselines: 
            logL= logL + self.log_likelihood_IJ(baseline,noise=True)
        return logL
    
    def OmegaGW(self,freqs):
        '''
        Subclasses should implement this funciton
        '''
        pass 
        
class PowerLawGWBLikelihood(GWBLikelihood):
    '''
    Power law GWB model
    
    Omega_GW(f|{A,alpha,fref=fixed}) = A * (f/fref)**alpha
    '''
    
    def __init__(self,baselines,fref=1):
        super().__init__(baselines,parameters={'A':None,'alpha':None})
        self.fref=fref
        
    def OmegaGW(self,freqs):
        A = self.parameters['A']
        alpha = self.parameters['alpha']
        return A*(freqs/self.fref)**alpha

class BasicGWBLikelihood(GWBLikelihood):
    '''
    Basic version of GWB Likelihood, in which the user 
    just provides Y_f, sigma_f, and freqs, and doesn't need
    to worry about the Baseline class
    '''
    def __init__(self, Y_f,var_f,freqs,parameters=None):
        baselines=[Baseline(None,Y_f,var_f,freqs)]
        super().__init__(baselines,parameters=parameters)
        
class BasicPowerLawGWBLikelihood(BasicGWBLikelihood):
    '''
    Power law GWB model
    '''
    
    def __init__(self,Y_f,var_f,freqs,fref=1):
        super().__init__(Y_f,var_f,freqs,parameters={'A':None,'alpha':None})
        self.fref=fref
        
    def OmegaGW(self,freqs):
        A = self.parameters['A']
        alpha = self.parameters['alpha']
        return A*(freqs/self.fref)**alpha